# Adding a new task to piepy

The **entire** surface for adding your own StimPy task: drop **two files** into `~/.piepy/paradigms/<task>/` and analyze. No edits to piepy's source, no wheel/visual code required.

We use a toy non-visual task **`tonetask`** (a tone + lick-to-report). Its parser is exercised on **synthetic** rawdata, so this notebook runs end-to-end *without* rig files.

**Assumes:** StimPy `.stimlog`/`.riglog` for *real* sessions, and `~/.piepy/config.json` `paths` pointed at your data (only needed for the real-data cells near the end).

In [ ]:
from pathlib import Path
from piepy.core.config import config

# the drop-in store (auto-created; default ~/.piepy/paradigms)
paradigms_dir = Path(config.paths["paradigms"][0])
task_dir = paradigms_dir / "tonetask"
task_dir.mkdir(parents=True, exist_ok=True)
print("paradigm store:", paradigms_dir)

## 1. `scheme.json` — session-name scheme + state-transition map (data, no code)

- **`naming_template`**: a regex with named groups; must capture `date` (YYMMDD) and `animalid`. The paradigm is taken from the folder name; any other named group lands in `extra`.
- **`state_transitions`**: maps StimPy's numbered state edges (`"old->new"`) to the names your parser uses.

In [ ]:
import json

scheme = {
    "naming_template": r"(?P<date>\d{6})_(?P<animalid>[^_]+)_tonetask__(?P<user>[^_]+)",
    "state_transitions": {
        "0->1": "trialstart",
        "1->2": "stimstart",
        "2->3": "responsewindow",
        "3->0": "trialend",
    },
}
(task_dir / "scheme.json").write_text(json.dumps(scheme, indent=2))
print((task_dir / "scheme.json").read_text())

## 2. `handler.py` — schema + parser (the only real code)

`TrialHandler` is the **channel-agnostic StimPy base**. Inside `get_trial` you call:

- `set_trial(trial_no, rawdata)` — windows every channel to the trial,
- `transition_time(name)` — elapsed time of a state transition,
- `rig_event(name)` — `(time, value)` events for a hardware channel (`lick`, `reward`, …).

Then assemble your trial dict and `register_paradigm(...)`. Transitions + naming come from `scheme.json`. **Note: no wheel/visual imports.**

In [ ]:
HANDLER_SRC = r'''
"""tonetask: a toy non-visual StimPy task (tone + lick-to-report).

Self-contained: schema + parser + registration. No wheel/visual imports -- it builds only on
the channel-agnostic StimPy primitives of the base TrialHandler.
"""
import patito as pt
import polars as pl
from piepy.core.trial import Trial, TrialHandler
from piepy.core.registry import register_paradigm


class ToneTrial(Trial):
    t_stimstart: int | None = pt.Field(default=None, dtype=pl.UInt64)
    response_latency: float | None = pt.Field(default=None, dtype=pl.Float64)
    lick_count: int | None = pt.Field(default=None, dtype=pl.Int64)
    outcome: str | None = pt.Field(default=None, dtype=pl.Utf8)


class ToneTrialHandler(TrialHandler):
    # validated at registration against scheme.json's state_transitions
    required_transitions = frozenset({"trialstart", "stimstart", "trialend"})

    def __init__(self):
        super().__init__()
        self.set_model(ToneTrial)

    def get_trial(self, trial_no, rawdata, return_as="dict"):
        self.init_trial()
        if not self.set_trial(trial_no, rawdata):   # window every channel to this trial
            return None
        t_stim = self.transition_time("stimstart")  # state event time
        self._trial["t_stimstart"] = t_stim
        licks = self.rig_event("lick")              # (time, value) hardware events, or None
        self._trial["lick_count"] = 0 if licks is None else int(len(licks))
        if licks is not None and t_stim is not None:
            after = [t for t in licks[:, 0] if t >= t_stim]
            self._trial["response_latency"] = float(after[0] - t_stim) if after else None
        self._trial["outcome"] = "lick" if self._trial["lick_count"] else "nolick"
        return self._update_and_return(return_as)


register_paradigm("tonetask", trial_handler_cls=ToneTrialHandler)
'''
(task_dir / "handler.py").write_text(HANDLER_SRC)
print("wrote", task_dir / "handler.py")

## 3. Auto-discovery

Resolving the paradigm **imports `handler.py` and reads `scheme.json` automatically** — no manual import needed.

In [ ]:
from piepy.core.registry import get_paradigm

spec = get_paradigm("tonetask")          # imports handler.py + reads scheme.json
run_cls = spec.session_cls.run_cls
print("session class :", spec.session_cls.__name__)
print("handler       :", run_cls.trial_handler_cls.__name__)
print("transitions   :", run_cls.state_transitions)

## 4. Run the parser on synthetic rawdata (no rig files needed)

`rawdata` is a dict of polars frames keyed by StimPy channel (`statemachine`, `lick`, …). Here is a 2-trial fake to prove the parser end-to-end.

In [ ]:
import polars as pl

handler = run_cls.trial_handler_cls()
rawdata = {
    "statemachine": pl.DataFrame(
        {
            "trialNo": [1, 1, 1, 2, 2, 2],
            "transition": ["trialstart", "stimstart", "trialend",
                           "trialstart", "stimstart", "trialend"],
            "elapsed": [0, 20, 80, 100, 120, 200],
        }
    ),
    "lick": pl.DataFrame({"duinotime": [25, 30, 140], "value": [1, 1, 1]}),
}

for t in (1, 2):
    print(f"trial {t}:", handler.get_trial(t, rawdata))

## 5. Analyze real sessions

With real StimPy data and `config.paths` set, the same paradigm runs through the standard entrypoints (these need your data, so they are commented out):

In [ ]:
# --- single session ---
# from piepy.core.registry import get_session_class
# sess = get_session_class("tonetask")("250618_M1_tonetask__bob", load_flag=False)
# df = sess.concatenate_runs("tonetask")

# --- many sessions (cohort) ---
# from piepy.core.hub import Hub
# hub = Hub("tonetask")
# hub.initialize(list_of_session_dirs, load_sessions=False)
# df = hub.data

## 6. Downstream analysis is paradigm-agnostic

`piepy.stats` and `piepy.fitting` operate on any paradigm's trial table — nothing task-specific. (Shown here on a *simulated* table, since `tonetask` has no simulator; the API is identical for your parsed `df`.)

In [ ]:
from piepy.simulations.session import simulate_session
from piepy.stats import aggregate, Median

demo = simulate_session(paradigm="detection", n_trials=400, seed=0)
aggregate(demo, group=["signed_contrast"], metrics=[Median("response_time")])

## Cleanup (optional)
Uncomment to remove the toy task folder from your paradigm store.

In [ ]:
# import shutil
# shutil.rmtree(task_dir)
# print("removed", task_dir)